# Model Comparison: YOLO vs. Image-Focused Models

One notebook, two independent halves:

1. **Run a new comparison** — our trained YOLO detector vs. one or more
   LLM/VLMs, on the same sample of held-out test images. Skip this section
   entirely if you just want to re-analyze a run you already have.
2. **Load a run for analysis** — scores and visualizes whatever's in
   `runs/llm/<run_name>/`, loaded fresh from disk. Re-run just this half as
   many times as you want without repeating any model inference.
3. **Descriptive vs. structured prompt (Gemini only)** — a small, unscored
   qualitative side-by-side: the same images, run through Gemini once with
   the free-text "site record" prompt and once with the structured JSON
   presence prompt everyone else in section 1/2 is scored on. There's no
   ground truth for prose, so this section is for eyeballing quality, not
   a metrics table.

For an unattended overnight batch run, use `scripts/compare_models.py`
(CLI) instead of section 1 — a notebook you have to babysit is the wrong
tool for that; both write to the same `runs/llm/` layout this notebook's
analysis half reads from.

Two scoring tiers, because LLMs and YOLO aren't directly comparable:
- **Presence**: "does this image contain a helmet?" — fair across every
  model, LLM or not.
- **Box-level**: actual IoU-matched precision/recall — meaningful only for
  YOLO, the one grounding-capable, non-LLM baseline. Every LLM/VLM entrant
  is judged on presence only, never box level — their coordinates would be
  guesses.

Run from the repo root.


In [ ]:
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import yaml

REPO_ROOT = Path.cwd()
MERGED_ROOT = REPO_ROOT / "data" / "merged"
LLM_RUNS_ROOT = REPO_ROOT / "runs" / "llm"

sys.path.insert(0, str(REPO_ROOT / "scripts"))
from compare_models import (  # noqa: E402
    DATASET_NAME,
    DEFAULT_YOLO_WEIGHTS,
    build_adapter,
    build_run_name,
    image_path_for as raw_image_path_for,
    sample_test_images,
)
from model_adapters import ADAPTERS, DEFAULT_PROMPT_TEMPLATE, render_prompt  # noqa: E402

CLASS_NAMES = yaml.safe_load(open(MERGED_ROOT / "data.yaml"))["names"]

## 1. Run a new comparison

Optional — skip to "Load a run for analysis" below if you already have a run to look at.

### Configuration — edit these and re-run

In [ ]:
N_IMAGES = 20  # -1 = every image in the test split, not a sample
MODELS = ["yolo", "ollama", "qwen3-vl", "gemma4", "minicpm-v"]  # available: yolo, ollama, qwen3-vl, gemma4, minicpm-v, claude, gemini
INCLUDE_CLOUD = False  # must be True to allow a cloud model (claude/gemini) to run —
                       # guards against accidental API spend, especially from
                       # a stray "Run All"
SEED = 42
NEW_RUN_NAME = None  # None = auto-built from timestamp/dataset/n/seed/models

# ollama, qwen3-vl, gemma4, and minicpm-v are all served by Ollama,
# just with different model tags — four different model families
# (LLaVA/Qwen/Gemma/MiniCPM) — see model_adapters.ADAPTERS for the exact
# tags (qwen3-vl:4b, gemma4:e4b, minicpm-v:8b — confirmed real
# Ollama library tags).
model_config = SimpleNamespace(
    yolo_weights=str(DEFAULT_YOLO_WEIGHTS),
    ollama_model=ADAPTERS["ollama"]["default_model"],
    qwen3_vl_model=ADAPTERS["qwen3-vl"]["default_model"],
    gemma4_model=ADAPTERS["gemma4"]["default_model"],
    minicpm_v_model=ADAPTERS["minicpm-v"]["default_model"],
    ollama_url="http://localhost:11434",
    claude_model=ADAPTERS["claude"]["default_model"],  # claude-haiku-4-5 — see accompanying note on why not Opus
    gemini_model=ADAPTERS["gemini"]["default_model"],  # gemini-3.6-flash — see model_adapters.GeminiAdapter
    prompt_template=DEFAULT_PROMPT_TEMPLATE,
)

### Preview the prompt

This is the literal text (with the real class list filled in) that gets sent to every chat-style model (ollama/qwen3-vl/gemma4/minicpm-v/claude) alongside each image. Edit `model_config.prompt_template` above — it needs `{class_list}` and `{json_shape}` placeholders — and re-run this cell to see the effect before committing to a full run.

In [ ]:
print(render_prompt(model_config.prompt_template, ADAPTERS["claude"]["cls"].queryable_classes))

### Sample test images

In [ ]:
unknown = [m for m in MODELS if m not in ADAPTERS]
if unknown:
    raise ValueError(f"Unknown model(s) {unknown}. Available: {list(ADAPTERS)}")

cloud_requested = [m for m in MODELS if ADAPTERS[m]["is_cloud"]]
if cloud_requested and not INCLUDE_CLOUD:
    raise ValueError(
        f"{cloud_requested} require INCLUDE_CLOUD = True (network calls to a paid API) — "
        "set it above and re-run this cell to confirm."
    )

sampled_files = sample_test_images(N_IMAGES, SEED)
print(f"Sampled {len(sampled_files)} test images (n_images={N_IMAGES}, seed={SEED})")

### Load models

In [ ]:
adapters = {}
for name in MODELS:
    print(f"Loading {name}...")
    t0 = time.perf_counter()
    adapters[name] = build_adapter(name, model_config)
    print(f"  ready in {time.perf_counter() - t0:.1f}s")

### Run predictions

This is the slow cell — cloud models add network latency, and every local
Ollama chat model needs a full generation call per image (unlike yolo,
which runs in a fraction of a second). Progress prints every 20 (image,
model) pairs.

In [ ]:
detection_rows = []
presence_rows = []
parse_failures = {name: 0 for name in MODELS}

total = len(sampled_files) * len(MODELS)
done = 0
t_start = time.perf_counter()

for file_stem in sampled_files:
    image_path = raw_image_path_for(file_stem)
    if image_path is None:
        print(f"warning: no image found for {file_stem}, skipping")
        continue

    for name, adapter in adapters.items():
        done += 1
        detections = adapter.predict(image_path)

        if detections is None:  # unparseable model output
            parse_failures[name] += 1
            for cls in adapter.queryable_classes:
                presence_rows.append(
                    {"file": file_stem, "model": name, "class_name": cls, "present": None, "parse_error": True}
                )
            continue

        present_classes = {d.class_name for d in detections if d.present}
        for cls in adapter.queryable_classes:
            presence_rows.append(
                {
                    "file": file_stem,
                    "model": name,
                    "class_name": cls,
                    "present": cls in present_classes,
                    "parse_error": False,
                }
            )
        for d in detections:
            if d.bbox is None and not d.present:
                continue  # a chat-model "false" isn't a detection row
            detection_rows.append(
                {
                    "file": file_stem,
                    "model": name,
                    "class_name": d.class_name,
                    "confidence": d.confidence,
                    "x1": d.bbox[0] if d.bbox else None,
                    "y1": d.bbox[1] if d.bbox else None,
                    "x2": d.bbox[2] if d.bbox else None,
                    "y2": d.bbox[3] if d.bbox else None,
                }
            )

        if done % 20 == 0 or done == total:
            elapsed = time.perf_counter() - t_start
            print(f"  {done}/{total} (image, model) pairs done, {elapsed:.0f}s elapsed")

print(f"\nDone in {time.perf_counter() - t_start:.0f}s")

### Save results

In [ ]:
new_run_name = NEW_RUN_NAME or build_run_name(N_IMAGES, SEED, MODELS)
new_run_dir = LLM_RUNS_ROOT / new_run_name
new_run_dir.mkdir(parents=True, exist_ok=True)

detections_path = new_run_dir / "detections.csv"
presence_path = new_run_dir / "presence.csv"
pd.DataFrame(detection_rows).to_csv(detections_path, index=False)
pd.DataFrame(presence_rows).to_csv(presence_path, index=False)

new_run_manifest = {
    "run_name": new_run_name,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "dataset_name": DATASET_NAME,
    "n_images_requested": N_IMAGES,
    "n_images_sampled": len(sampled_files),
    "seed": SEED,
    "models": MODELS,
    "queryable_classes": {name: adapters[name].queryable_classes for name in MODELS},
    "supports_grounding": {name: adapters[name].supports_grounding for name in MODELS},
    "prompt_template": model_config.prompt_template,
    "config": vars(model_config),
    "parse_failures": parse_failures,
    "sampled_files": sampled_files,
}
manifest_path = new_run_dir / "run_manifest.json"
with manifest_path.open("w") as f:
    json.dump(new_run_manifest, f, indent=2)

print(f"Wrote:\n  {manifest_path}\n  {detections_path} ({len(detection_rows)} rows)\n  {presence_path} ({len(presence_rows)} rows)")
if any(parse_failures.values()):
    print(f"parse failures: {parse_failures}")
print(f"\nScroll down to \"Load a run for analysis\" — RUN_NAME=None there will pick this run up automatically.")

## 2. Load a run for analysis

Independent of section 1 — reads from disk, so re-running just this half
never repeats any model inference. `RUN_NAME = None` picks the most
recently created run (the one you just made above, if any); set it
explicitly to analyze an older run instead.

In [ ]:
RUN_NAME = None
if RUN_NAME is None:
    candidates = sorted((p for p in LLM_RUNS_ROOT.iterdir() if p.is_dir()), key=lambda p: p.stat().st_mtime) if LLM_RUNS_ROOT.exists() else []
    if not candidates:
        raise SystemExit(
            "No runs found under runs/llm/ — run section 1 above, or "
            "`python scripts/compare_models.py` (CLI, defaults to 20 images)."
        )
    RUN_DIR = candidates[-1]
else:
    RUN_DIR = LLM_RUNS_ROOT / RUN_NAME

manifest = json.loads((RUN_DIR / "run_manifest.json").read_text())
presence = pd.read_csv(RUN_DIR / "presence.csv")
detections = pd.read_csv(RUN_DIR / "detections.csv")

print(f"Run: {RUN_DIR.name}")
print(f"Models: {manifest['models']}")
print(f"Images: {manifest['n_images_sampled']}")
if any(manifest["parse_failures"].values()):
    print(f"Parse failures (excluded below): {manifest['parse_failures']}")

In [ ]:
def gt_classes_for(file):
    """Every class actually present in this image, from its ground-truth
    label file (not from any model's prediction)."""
    label_path = MERGED_ROOT / "test" / "labels" / f"{file}.txt"
    classes = set()
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            if line.strip():
                classes.add(CLASS_NAMES[int(line.split()[0])])
    return classes


def gt_boxes_for(file, class_name):
    label_path = MERGED_ROOT / "test" / "labels" / f"{file}.txt"
    boxes = []
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            if not line.strip():
                continue
            parts = line.split()
            if CLASS_NAMES[int(parts[0])] != class_name:
                continue
            xc, yc, w, h = map(float, parts[1:5])
            boxes.append((xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2))
    return boxes


def iou(b1, b2):
    ax1, ay1, ax2, ay2 = b1
    bx1, by1, bx2, by2 = b2
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    a1 = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    a2 = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0


gt_presence_rows = []
for file in presence["file"].unique():
    present = gt_classes_for(file)
    for c in CLASS_NAMES:
        gt_presence_rows.append({"file": file, "class_name": c, "gt_present": c in present})
gt_presence = pd.DataFrame(gt_presence_rows)

## Presence-level metrics

Precision/recall/F1 per model per class, restricted to the classes each model was actually asked about (a `no-X` class isn't fair to score against a model that can't express negation).

In [ ]:
scored = presence[~presence["parse_error"]].merge(gt_presence, on=["file", "class_name"], how="left")


def prf(group):
    tp = ((group["present"] == True) & (group["gt_present"] == True)).sum()
    fp = ((group["present"] == True) & (group["gt_present"] == False)).sum()
    fn = ((group["present"] == False) & (group["gt_present"] == True)).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else float("nan")
    return pd.Series({"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1})


presence_metrics = scored.groupby(["model", "class_name"], group_keys=True).apply(prf, include_groups=False)
presence_metrics.style.background_gradient(cmap="RdYlGn", subset=["precision", "recall", "f1"], vmin=0, vmax=1).format(
    {"precision": "{:.2f}", "recall": "{:.2f}", "f1": "{:.2f}"}
)

## Box-level metrics (grounding-capable models only)

IoU >= 0.5 greedy matching, per class. Only computed for models whose `run_manifest.json` entry says `supports_grounding`, and only on the classes that model was actually queried on.

In [ ]:
def box_prf(model_name, files, classes, iou_thresh=0.5):
    det_sub = detections[detections["model"] == model_name]
    rows = []
    for class_name in classes:
        tp = fp = fn = 0
        cls_dets = det_sub[det_sub["class_name"] == class_name]
        for f in files:
            gts = gt_boxes_for(f, class_name)
            preds = cls_dets[cls_dets["file"] == f][["x1", "y1", "x2", "y2"]].dropna().values.tolist()
            matched_gt = set()
            for p in preds:
                best_iou, best_j = 0, -1
                for j, g in enumerate(gts):
                    if j in matched_gt:
                        continue
                    v = iou(p, g)
                    if v > best_iou:
                        best_iou, best_j = v, j
                if best_iou >= iou_thresh:
                    tp += 1
                    matched_gt.add(best_j)
                else:
                    fp += 1
            fn += len(gts) - len(matched_gt)
        precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
        recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        rows.append({"model": model_name, "class_name": class_name, "tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall})
    return pd.DataFrame(rows)


grounding_models = [m for m in manifest["models"] if manifest["supports_grounding"].get(m)]
files = presence["file"].unique()
box_metrics = pd.concat(
    [box_prf(m, files, manifest["queryable_classes"][m]) for m in grounding_models],
    ignore_index=True,
).set_index(["model", "class_name"])

box_metrics.style.background_gradient(cmap="RdYlGn", subset=["precision", "recall"], vmin=0, vmax=1).format(
    {"precision": "{:.2f}", "recall": "{:.2f}"}
)

## Side-by-side visual comparison

Ground truth (green) vs. each model's predictions (red), for a random sample.

In [ ]:
def image_path_for(file):
    for ext in (".jpg", ".jpeg", ".png", ".bmp"):
        p = MERGED_ROOT / "test" / "images" / f"{file}{ext}"
        if p.exists():
            return p
    return None


def draw(file, model=None):
    """model=None draws ground truth; otherwise that model's predicted
    boxes for the classes it was queried on."""
    path = image_path_for(file)
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if model is None:
        label_path = MERGED_ROOT / "test" / "labels" / f"{file}.txt"
        color = (0, 200, 0)
        for line in label_path.read_text().splitlines():
            if not line.strip():
                continue
            parts = line.split()
            cid = int(parts[0])
            xc, yc, bw, bh = (float(v) for v in parts[1:5])
            x1, y1, x2, y2 = int((xc - bw / 2) * w), int((yc - bh / 2) * h), int((xc + bw / 2) * w), int((yc + bh / 2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, CLASS_NAMES[cid], (x1, max(y1 - 5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    else:
        color = (255, 0, 0)
        rows = detections[(detections["model"] == model) & (detections["file"] == file)].dropna(subset=["x1"])
        for _, row in rows.iterrows():
            x1, y1, x2, y2 = int(row["x1"] * w), int(row["y1"] * h), int(row["x2"] * w), int(row["y2"] * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, row["class_name"], (x1, max(y1 - 5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img


sample_files = pd.Series(presence["file"].unique()).sample(min(4, presence["file"].nunique()), random_state=0)
columns = ["ground truth"] + grounding_models
fig, axes = plt.subplots(len(sample_files), len(columns), figsize=(5 * len(columns), 5 * len(sample_files)))
if len(sample_files) == 1:
    axes = axes.reshape(1, -1)
for row_ax, file in zip(axes, sample_files):
    row_ax[0].imshow(draw(file))
    row_ax[0].set_title(f"ground truth\n{file}", fontsize=7)
    row_ax[0].axis("off")
    for ax, model in zip(row_ax[1:], grounding_models):
        ax.imshow(draw(file, model))
        ax.set_title(model, fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

## Biggest disagreements

Images where a model's presence call diverges most from YOLO — worth eyeballing to see who's actually right.

In [ ]:
if "yolo" not in manifest["models"]:
    print("yolo wasn't in this run — skipping (disagreement is measured against it as the reference).")
else:
    yolo_presence = scored[scored["model"] == "yolo"][["file", "class_name", "present"]].rename(columns={"present": "yolo_present"})
    disagreement_rows = []
    for model in manifest["models"]:
        if model == "yolo":
            continue
        other = scored[scored["model"] == model][["file", "class_name", "present"]].rename(columns={"present": "other_present"})
        joined = other.merge(yolo_presence, on=["file", "class_name"])
        joined = joined[joined["yolo_present"] != joined["other_present"]]
        joined["model"] = model
        disagreement_rows.append(joined)

    if disagreement_rows:
        disagreements = pd.concat(disagreement_rows, ignore_index=True)
        per_file_disagreement = disagreements.groupby("file").size().sort_values(ascending=False)
        print(f"{len(disagreements)} (model, class) disagreements with yolo across {presence['file'].nunique()} images")

        top_files = per_file_disagreement.head(4).index.tolist()
        fig, axes = plt.subplots(len(top_files), len(columns), figsize=(5 * len(columns), 5 * len(top_files)))
        if len(top_files) == 1:
            axes = axes.reshape(1, -1)
        for row_ax, file in zip(axes, top_files):
            row_ax[0].imshow(draw(file))
            row_ax[0].set_title(f"ground truth\n{file}", fontsize=7)
            row_ax[0].axis("off")
            for ax, model in zip(row_ax[1:], grounding_models):
                ax.imshow(draw(file, model))
                ax.set_title(model, fontsize=9)
                ax.axis("off")
        plt.tight_layout()
        plt.show()

        print(disagreements[disagreements["file"].isin(top_files)].sort_values("file").to_string(index=False))
    else:
        print("No disagreements found.")

## 3. Descriptive vs. structured prompt (Gemini)

Independent of sections 1/2 above. Sends **the same small sample of images**
(from `data/merged/test/images`, via `sample_test_images()` — identical
sampling to section 1) through Gemini twice:

1. **The new descriptive prompt** — one or two plain sentences for a site
   record: scene, rough headcount + activity, setting, and a safety/
   compliance/risk assessment. Free text, no schema.
2. **The existing structured prompt** — the same `DEFAULT_PROMPT_TEMPLATE`
   every model in section 1/2 is scored on (strict JSON, one bool per
   class).

Deliberately a **small, unscored** sample — there's no ground-truth prose to
check the descriptive prompt's output against, so this is for reading side
by side, not computing precision/recall.

Needs `GEMINI_API_KEY` in the environment (never hardcode it in this
notebook). Run the cell below to generate fresh data, or skip it and the
load cell after picks up the most recent `runs/llm/*/prompt_comparison.csv`
on disk — including ones from a CLI run of `scripts/gemini_prompt_comparison.py`.


In [ ]:
# Generate fresh Gemini prompt-comparison data — same test-set sampling as
# section 1 (sample_test_images, seed 42), just a smaller N_IMAGES since this
# is an unscored qualitative sample, not a metrics run. Skip this cell to
# just load an existing runs/llm/*/prompt_comparison.csv in the next cell.
N_IMAGES = 10
SEED = 42

PROMPT = (
    "Describe this construction site photograph in one or two plain sentences "
    "for a site record: what the scene is, roughly how many people are visible "
    "and what they appear to be doing, and the setting. Describe only what is "
    "visible. Assess safety, compliance or risk. No preamble, no bullet "
    "points."
)

gemini_adapter = ADAPTERS["gemini"]["cls"](model=ADAPTERS["gemini"]["default_model"])  # reads GEMINI_API_KEY from the environment
structured_prompt = render_prompt(DEFAULT_PROMPT_TEMPLATE, gemini_adapter.queryable_classes)
prompts = {"1_new_descriptive": PROMPT, "2_existing_structured": structured_prompt}

sampled_for_prompt = sample_test_images(N_IMAGES, SEED)
print(f"Sampled {len(sampled_for_prompt)} test images (n_images={N_IMAGES}, seed={SEED})")

prompt_rows = []
for file_stem in sampled_for_prompt:
    image_path = raw_image_path_for(file_stem)
    if image_path is None:
        print(f"warning: no image found for {file_stem}, skipping")
        continue
    for prompt_name, prompt_text in prompts.items():
        text = gemini_adapter.describe(image_path, prompt_text)
        prompt_rows.append({"file": file_stem, "prompt_name": prompt_name, "response_text": text.strip()})
        print(f"  {file_stem} / {prompt_name}: {text.strip()[:80]!r}")

new_prompt_run_name = f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}_{DATASET_NAME}_gemini_prompt_comparison_n{len(sampled_for_prompt)}_seed{SEED}"
new_prompt_run_dir = LLM_RUNS_ROOT / new_prompt_run_name
new_prompt_run_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame(prompt_rows).to_csv(new_prompt_run_dir / "prompt_comparison.csv", index=False)
print(f"\nWrote {new_prompt_run_dir / 'prompt_comparison.csv'} ({len(prompt_rows)} rows)")


In [ ]:
PROMPT_RUN_NAME = None  # None = most recent runs/llm/*/prompt_comparison.csv
if PROMPT_RUN_NAME is None:
    candidates = sorted(
        (p for p in LLM_RUNS_ROOT.iterdir() if p.is_dir() and (p / "prompt_comparison.csv").exists()),
        key=lambda p: p.stat().st_mtime,
    ) if LLM_RUNS_ROOT.exists() else []
    if not candidates:
        raise SystemExit(
            "No prompt_comparison.csv found under runs/llm/ — run "
            "`python scripts/gemini_prompt_comparison.py` first."
        )
    PROMPT_RUN_DIR = candidates[-1]
else:
    PROMPT_RUN_DIR = LLM_RUNS_ROOT / PROMPT_RUN_NAME

prompt_comparison = pd.read_csv(PROMPT_RUN_DIR / "prompt_comparison.csv")
print(f"Run: {PROMPT_RUN_DIR.name}")
print(f"Images: {prompt_comparison['file'].nunique()}")
prompt_comparison.head()


In [ ]:
for file, group in prompt_comparison.groupby("file"):
    image_path = raw_image_path_for(file)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={"width_ratios": [1, 1.3]})
    if image_path is not None:
        axes[0].imshow(cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB))
    axes[0].set_title(file, fontsize=8)
    axes[0].axis("off")

    lines = []
    for _, row in group.sort_values("prompt_name").iterrows():
        lines.append(f"[{row['prompt_name']}]\n{row['response_text']}\n")
    axes[1].axis("off")
    axes[1].text(0, 1, "\n".join(lines), fontsize=9, va="top", wrap=True)
    plt.tight_layout()
    plt.show()
